# Transfer Learning and Fine-Tuning for Images

Notebook for **Chapter 7** of the *dl* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dl/ch07.html

Tip: change a number, predict the output, then run the cell again.

## 7.1 Why Transfer Learning Works

In [ ]:
import numpy as np
rng=np.random.default_rng(0)
D=60; L=3
M=rng.normal(size=(L,D))
def make(n):
    z=rng.normal(size=(n,L)); return z@M+5.0*rng.normal(size=(n,D)), z
xs,zs=make(2000); ys=(zs>0).astype(float)
def target(n):
    x,z=make(n); return x,((z[:,0]+z[:,1]-z[:,2])>0).astype(float)
sig=lambda a:1/(1+np.exp(-a))
def init(h=8,out=1,seed=1):
    r=np.random.default_rng(seed)
    return {'W1':r.normal(size=(D,h))*np.sqrt(2/D),'b1':np.zeros(h),
            'W2':r.normal(size=(h,out))*np.sqrt(1/h),'b2':np.zeros(out)}
def fwd(p,x):
    h=np.tanh(x@p['W1']+p['b1']); return h,sig(h@p['W2']+p['b2'])
def train(p,x,y,epochs,lr,freeze=False):
    n=len(x)
    for e in range(epochs):
        h,o=fwd(p,x); d=(o-y)/n
        p['W2']-=lr*(h.T@d); p['b2']-=lr*d.sum(0)
        if not freeze:
            dh=(d@p['W2'].T)*(1-h**2)
            p['W1']-=lr*(x.T@dh); p['b1']-=lr*dh.sum(0)
    return p
def acc(p,x,y): return ((fwd(p,x)[1]>0.5)==y).mean()
src=train(init(8,3,seed=1),xs,ys,3000,0.3)          # "pretraining"
print("source acc",[round(float(((fwd(src,xs)[1][:,i]>0.5)==ys[:,i]).mean()),3) for i in range(3)])
xt_te,yt_te=target(5000); yte=yt_te[:,None]
for ntr in (10,20,50,500):
  out={'scratch':[],'frozen':[],'finetune':[]}
  for rep in range(20):
    xt,yt=target(ntr); yt=yt[:,None]
    p=train(init(8,1,seed=10+rep),xt,yt,2000,0.3); out['scratch'].append(acc(p,xt_te,yte))
    q={'W1':src['W1'].copy(),'b1':src['b1'].copy(),'W2':np.zeros((8,1)),'b2':np.zeros(1)}
    q=train(q,xt,yt,2000,0.3,freeze=True); out['frozen'].append(acc(q,xt_te,yte))
    r={k:v.copy() for k,v in q.items()}
    r=train(r,xt,yt,200,0.02); out['finetune'].append(acc(r,xt_te,yte))
  print(ntr,{k:round(float(np.mean(v)),3) for k,v in out.items()})

source acc [0.875, 0.85, 0.888]
10 {'scratch': 0.579, 'frozen': 0.628, 'finetune': 0.628}
20 {'scratch': 0.6, 'frozen': 0.644, 'finetune': 0.646}
50 {'scratch': 0.661, 'frozen': 0.708, 'finetune': 0.71}
500 {'scratch': 0.752, 'frozen': 0.744, 'finetune': 0.748}


## 7.2 The Fine-Tuning Recipe: Freezing, Learning Rates, Normalization

In [ ]:
import numpy as np
fc_old=512*1000+1000; fc_new=512*5+5
print("old fc",fc_old,"new head",fc_new)
total=11_689_512
print("share trainable if only head:", round(fc_new/(total-fc_old+fc_new)*100,3),"%")
base=1e-3
for i,name in enumerate(["stem+layer1","layer2","layer3","layer4","head"]):
    print(f"{name:12s} {base*(0.3**(4-i)):.2e}")

old fc 513000 new head 2565
share trainable if only head: 0.023 %
stem+layer1  8.10e-06
layer2       2.70e-05
layer3       9.00e-05
layer4       3.00e-04
head         1.00e-03


In [ ]:
import torch, torch.nn as nn
from torchvision import models

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)  # downloads pretrained weights
for p in model.parameters():
    p.requires_grad = False                 # freeze the whole backbone
model.fc = nn.Linear(512, 5)                # new head; its parameters are trainable by default

opt = torch.optim.AdamW(model.fc.parameters(), lr=1e-3)      # Phase 1: head only
# ... train a few epochs ...

for p in model.layer4.parameters():         # Phase 2: unfreeze the last stage
    p.requires_grad = True
opt = torch.optim.AdamW([
    {"params": model.layer4.parameters(), "lr": 3e-4},
    {"params": model.fc.parameters(),     "lr": 1e-3},
], weight_decay=1e-2)

## 7.3 Data Augmentation and Dataset Care

In [ ]:
import numpy as np
img = np.arange(36).reshape(6, 6).astype(float)
flip = img[:, ::-1]
crop = img[1:5, 2:6]
print(img[0]); print(flip[0]); print(crop)
bright = np.clip(img * 1.2 + 10, 0, 255); print(bright[0])
print(img.mean(), flip.mean())

[0. 1. 2. 3. 4. 5.]
[5. 4. 3. 2. 1. 0.]
[[ 8.  9. 10. 11.]
 [14. 15. 16. 17.]
 [20. 21. 22. 23.]
 [26. 27. 28. 29.]]
[10.  11.2 12.4 13.6 14.8 16. ]


In [ ]:
import albumentations as A
train_tf = A.Compose([
    A.RandomResizedCrop(size=(224, 224), scale=(0.7, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
# validation: only resize + normalize, no random changes

## 7.4 More Ways to Adapt: Layer-Wise Rates, Imbalance, Mixing and Few-Shot Probes

Sections 7.1 to 7.3 gave you the core recipe: freeze, then unfreeze with a small learning rate, and augment. Real projects add four more tools. Each has a small worked example below.

In [ ]:
import numpy as np
# discriminative learning rates: layer groups from the head back to the stem
groups = ["head", "block4", "block3", "block2", "stem"]
base_lr, decay = 1e-3, 0.3
for g, k in zip(groups, range(len(groups))):
    print(f"{g:7s} lr {base_lr * decay**k:.2e}")

# class weights for an imbalanced set: 900 healthy, 90 mild, 10 severe
counts = np.array([900, 90, 10])
w_inv = counts.sum() / (len(counts) * counts)
print("inverse-frequency weights", np.round(w_inv, 3))
# weighted loss for one wrongly handled example per class, p(true class) = 0.5
loss = -np.log(0.5)
print("loss per example", round(loss, 3), "weighted", np.round(w_inv * loss, 3))
print("total weight mass per class", np.round(w_inv * counts, 1))

# mixup with lambda = 0.7 on two one-hot labels and two tiny "images"
lam = 0.7
a, b = np.array([[1.0, 0.0], [0.0, 1.0]]), np.array([[0.0, 4.0], [2.0, 0.0]])
ya, yb = np.array([1, 0, 0]), np.array([0, 0, 1])
print("mixed image\n", lam * a + (1 - lam) * b)
print("mixed label", lam * ya + (1 - lam) * yb)

# cutmix: area of the pasted patch sets lambda
H = W = 32; ph, pw = 16, 8
lam_cm = 1 - ph * pw / (H * W)
print("cutmix lambda", lam_cm, " label", np.round(lam_cm * ya + (1 - lam_cm) * yb, 3))

head    lr 1.00e-03
block4  lr 3.00e-04
block3  lr 9.00e-05
block2  lr 2.70e-05
stem    lr 8.10e-06
inverse-frequency weights [ 0.37   3.704 33.333]
loss per example 0.693 weighted [ 0.257  2.567 23.105]
total weight mass per class [333.3 333.3 333.3]
mixed image
 [[0.7 1.2]
 [0.6 0.7]]
mixed label [0.7 0.  0.3]
cutmix lambda 0.875  label [0.875 0.    0.125]


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
d, classes = 20, 3
centres = rng.normal(0, 1.0, (classes, d))          # frozen-feature class centres (synthetic)
def sample(n_per):
    X = np.vstack([centres[c] + rng.normal(0, 2.0, (n_per, d)) for c in range(classes)])
    y = np.repeat(np.arange(classes), n_per)
    return X, y
def fit_probe(X, y, epochs=300, lr=0.1, l2=1e-2):
    W = np.zeros((d, classes)); b = np.zeros(classes)
    Y = np.eye(classes)[y]
    for _ in range(epochs):
        z = X @ W + b; z -= z.max(1, keepdims=True)
        p = np.exp(z); p /= p.sum(1, keepdims=True)
        g = (p - Y) / len(X)
        W -= lr * (X.T @ g + l2 * W); b -= lr * g.sum(0)
    return W, b
Xte, yte = sample(500)
for n in (2, 5, 20, 100):
    accs = []
    for seed in range(5):
        rng = np.random.default_rng(100 + seed)
        Xtr, ytr = sample(n)
        W, b = fit_probe(Xtr, ytr)
        accs.append(((Xte @ W + b).argmax(1) == yte).mean())
    print(f"{n:4d} examples per class: linear-probe test accuracy {np.mean(accs):.3f} +/- {np.std(accs):.3f}")

   2 examples per class: linear-probe test accuracy 0.617 +/- 0.070
   5 examples per class: linear-probe test accuracy 0.690 +/- 0.061
  20 examples per class: linear-probe test accuracy 0.799 +/- 0.025
 100 examples per class: linear-probe test accuracy 0.863 +/- 0.005


## Chapter recap

## Practice set

**1.** (easy) `Linear(1024, 20)` has how many parameters? **Answer:** `1024 × 20 + 20 = 20,500`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dl/ch07.html